# Sentiment analysis model on reviews which tells us whether the review is positive, negative, neutral or mixed.

### Note: The dataset is created using AI, since i cant find someone with four labels like positive, negative, mixed and neutral, so there will be some issues but i will do my best to make it better , so that the model works better.

In [101]:
# import all required libraries

#import libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
import nltk
import re
import joblib

#import data preprocessing dependencies
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from textblob import TextBlob
from sklearn.preprocessing import LabelEncoder
import contractions


#import train_test_split
from sklearn.model_selection import train_test_split

#import evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#import warnings to ignore any warnings
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package stopwords to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Saif Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Saif Ullah\AppData\Roaming\nltk_data...
[nltk_d

In [102]:
# load the dataset
df = pd.read_csv('sentiment_dataset.csv')

In [103]:
#check the first 5 rows of the dataset
df.head()

,review,sentiment
0,My vacuum cleaner broke on day three. Returnin...,negative
1,My old one broke so I replaced it with this ca...,positive
2,Nothing about this backpack matches what was a...,negative
3,The protein powder makes a horrible screeching...,negative
4,The protein powder does what it's supposed to....,neutral


In [104]:
#check the shape of the dataset
df.shape

(5000, 2)

In [105]:
# check the label column
df['sentiment'].value_counts()

sentiment
positive    1500
negative    1300
neutral     1100
mixed       1100
Name: count, dtype: int64

In [106]:
#visualize the label distribution
sns.countplot(x='sentiment', data=df)
plt.title('Label Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

In [107]:
# chek for null values
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [108]:
#check for duplicate values
df.duplicated().sum()

np.int64(0)

#### since our dataset doesnt contain any null values and duplicate values we can proceed with data preprocessing

In [109]:
# split the dataset into features and labels
X = df.drop('sentiment', axis=1)
y = df['sentiment']

In [110]:
# check the shape of features and labels
print("Shape of features:", X.shape)
print("Shape of labels:", y.shape)

Shape of features: (5000, 1)
Shape of labels: (5000,)


In [111]:
#since the labels are categorical, we need to convert them into numerical values
# convert the labels into numerical values
le = LabelEncoder()
y = le.fit_transform(y)


In [112]:
# Assuming 'le' is the name of your LabelEncoder object
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']

In [113]:
# check the labels after encoding
print("Encoded labels:", np.unique(y))

Encoded labels: [0 1 2 3]


In [114]:
#we will remove tags and special characters from the review column
import re
def remove_tags(text):
    return re.sub(r'<,.*?>', '', text)
def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

In [115]:
# aplly the functions to the review column
df['review'] = df['review'].apply(remove_tags)
df['review'] = df['review'].apply(remove_special_characters)

In [116]:
# using a function that will do the following:
# 1. expand contractions
# 2. remove tags
# 3. remove special characters
# 4. convert to lowercase
# 5. remove extra spaces

def clean_text(text):
    # Expand contractions, such as "don't" to "do not"
    text = contractions.fix(text)
    # Remove punctuation (except ! and ?)
    text = re.sub(r"[.,:;()\[\]{}]", "", text)
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [117]:
# Apply to df['review']
df['review'] = df['review'].apply(clean_text)

In [118]:
df.head()

,review,sentiment
0,my vacuum cleaner broke on day three returning...,negative
1,my old one broke so i replaced it with this ca...,positive
2,nothing about this backpack matches what was a...,negative
3,the protein powder makes a horrible screeching...,negative
4,the protein powder does what its supposed to t...,neutral


In [119]:
# now we will apply tokenization, stopword removal, and lemmatization to the review column

# Step 1: Tokenization
def tokenize(text):
    return nltk.word_tokenize(text)

# apply tokenization to the review column
df['review'] = df['review'].apply(tokenize)

In [120]:
df['review'].head(20)

0     [my, vacuum, cleaner, broke, on, day, three, r...
1     [my, old, one, broke, so, i, replaced, it, wit...
2     [nothing, about, this, backpack, matches, what...
3     [the, protein, powder, makes, a, horrible, scr...
4     [the, protein, powder, does, what, its, suppos...
5     [bought, the, wireless, earbuds, for, a, speci...
6     [the, smartwatch, came, with, everything, i, n...
7     [the, headphones, is, lighter, than, i, expect...
8     [i, use, the, wireless, earbuds, daily, and, f...
9     [my, friend, recommended, this, gaming, mouse,...
10    [not, going, to, lie, this, laptop, is, genuin...
11    [just, what, i, needed, the, coffee, maker, fi...
12    [i, rarely, write, reviews, but, this, laptop,...
13    [i, use, the, coffee, maker, regularly, no, is...
14    [love, the, design, of, the, monitor, hate, th...
15    [bought, this, blender, as, a, gift, and, they...
16    [the, phone, case, is, a, threestar, product, ...
17    [the, air, fryer, is, lighter, than, i, ex

In [121]:
# step 2: lemmatization with POS tagging, that works better than lemmatization without POS tagging
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_tokens_with_pos(tokens):
    pos_tags = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]

#apply lemmatization with POS tagging to the review column
df['review'] = df['review'].apply(lemmatize_tokens_with_pos)


In [122]:
# check the review column after lemmatization
df['review'].head(20)

0     [my, vacuum, clean, break, on, day, three, ret...
1     [my, old, one, break, so, i, replace, it, with...
2     [nothing, about, this, backpack, match, what, ...
3     [the, protein, powder, make, a, horrible, scre...
4     [the, protein, powder, do, what, it, suppose, ...
5     [buy, the, wireless, earbuds, for, a, specific...
6     [the, smartwatch, come, with, everything, i, n...
7     [the, headphone, be, light, than, i, expect, b...
8     [i, use, the, wireless, earbuds, daily, and, f...
9     [my, friend, recommend, this, gaming, mouse, a...
10    [not, go, to, lie, this, laptop, be, genuinely...
11    [just, what, i, need, the, coffee, maker, fit,...
12    [i, rarely, write, review, but, this, laptop, ...
13    [i, use, the, coffee, maker, regularly, no, is...
14    [love, the, design, of, the, monitor, hate, th...
15    [buy, this, blender, a, a, gift, and, they, ab...
16    [the, phone, case, be, a, threestar, product, ...
17    [the, air, fryer, be, light, than, i, expe

In [123]:
# step 2: Stopword removal
stop_words = set(stopwords.words('english'))
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

#apply stopword removal to the review column
df['review'] = df['review'].apply(remove_stopwords)

In [124]:
#check the review column after stopword removal
df['review'].head(20)

0     [vacuum, clean, break, day, three, return, imm...
1     [old, one, break, replace, cast, iron, pan, on...
2     [nothing, backpack, match, advertised, dimensi...
3     [protein, powder, make, horrible, screech, sou...
4        [protein, powder, suppose, people, need, know]
5     [buy, wireless, earbuds, specific, task, handl...
6     [smartwatch, come, everything, need, box, miss...
7     [headphone, light, expect, feel, solid, wellmade]
8     [use, wireless, earbuds, daily, feel, nothing,...
9     [friend, recommend, gaming, mouse, pass, recom...
10    [go, lie, laptop, genuinely, incredible, use, ...
11    [need, coffee, maker, fit, perfectly, daily, r...
12    [rarely, write, review, laptop, earn, one, rea...
13    [use, coffee, maker, regularly, issue, excitem...
14                 [love, design, monitor, hate, price]
15    [buy, blender, gift, absolutely, love, look, e...
16    [phone, case, threestar, product, one, fivesta...
17    [air, fryer, light, expect, feel, solid, w

In [125]:
# train test split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['review'], y, test_size=0.2, random_state=42, stratify=y)

#### since we have two vectorizer (count and TFIDF ) and three Algorithms (logistics, SVC and Navie Byes) we will use pipeline in order to see which one works better and then we will use it.

In [126]:
# we will use both CountVectorizer and TfidfVectorizer to convert the text data into numerical data
vectorizers = {
    "CountVectorizer": CountVectorizer(),
    "TFIDFVectorizer": TfidfVectorizer()
}

# we will use all three models to train and evaluate the performance of the models
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVC": SVC(),
    "NaiveBayes": MultinomialNB()
}

# we will pick the best model based on accuracy score
best_acc = 0
best_model = None
best_vectorizer = None
best_combo = None

# convert token lists back to text for vectorizers
X_train_text = X_train.apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
X_test_text = X_test.apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

for vec_name, vect in vectorizers.items():
    # Fit vectorizer
    X_train_vec = vect.fit_transform(X_train_text)
    X_test_vec = vect.transform(X_test_text)
    
    for model_name, model in models.items():
        # Train
        model.fit(X_train_vec, y_train)
        # Predict
        y_pred = model.predict(X_test_vec)
        # Accuracy
        acc = accuracy_score(y_test, y_pred)
        print(f"{vec_name} + {model_name} Accuracy: {acc:.4f}")
        
        # Save best
        if acc > best_acc:
            best_acc = acc
            best_model = model
            best_vectorizer = vect
            best_combo = f"{vec_name} + {model_name}"

print(f"\nBest combination: {best_combo} with Accuracy: {best_acc:.4f}")

CountVectorizer + LogisticRegression Accuracy: 1.0000
CountVectorizer + SVC Accuracy: 1.0000
CountVectorizer + NaiveBayes Accuracy: 0.9930
TFIDFVectorizer + LogisticRegression Accuracy: 1.0000
TFIDFVectorizer + SVC Accuracy: 1.0000
TFIDFVectorizer + NaiveBayes Accuracy: 0.9940

Best combination: CountVectorizer + LogisticRegression with Accuracy: 1.0000


In [127]:
# Save best model and vectorizer
joblib.dump(best_model, "sentiment_classifier_model.pkl")
joblib.dump(best_vectorizer, "best_vectorizer.pkl")
print("Best model and vectorizer saved!")

Best model and vectorizer saved!


In [128]:
#evaluate the best model using accuracy, precision, recall, and f1-score
y_pred = best_model.predict(best_vectorizer.transform(X_test_text))
print("Classification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       220
           1       1.00      1.00      1.00       260
           2       1.00      1.00      1.00       220
           3       1.00      1.00      1.00       300

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [129]:
# visualize the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [130]:
#predict the sentiment of a new review
def predict_sentiment(review):
    # load the model and vectorizer
    model = joblib.load('sentiment_classifier_model.pkl')
    vectorizer = joblib.load('best_vectorizer.pkl')
    label_encoder = joblib.load('label_encoder.pkl')
    
    # preprocess the review
    review = clean_text(review)
    review = tokenize(review)
    review = lemmatize_tokens_with_pos(review)
    review = remove_stopwords(review)
    
    # convert the review back to text for vectorization
    review_text = " ".join(review)
    
    # vectorize the review
    review_vec = vectorizer.transform([review_text])
    
    # predict the sentiment
    pred_label_num = model.predict(review_vec)[0]
    
    # convert the numerical label back to original label
    pred_label = label_encoder.inverse_transform([pred_label_num])[0]
    
    return pred_label

In [137]:
# test the function with a new review
new_review = "the product was not good"
predicted_sentiment = predict_sentiment(new_review)
print(f"Review: {new_review}")
print(f"Predicted Sentiment: {predicted_sentiment}")

Review: the product was not good
Predicted Sentiment: mixed
